In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.instacart_silver;

-- Silver: aisles (dimension) - trimmed, deduplicated
CREATE OR REPLACE TABLE instacart_silver.aisles AS
SELECT
  aisle_id,
  TRIM(aisle) AS aisle
FROM instacart.aisles
WHERE aisle_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY aisle_id ORDER BY aisle) = 1;

-- Silver: departments (dimension) - trimmed, deduplicated
CREATE OR REPLACE TABLE instacart_silver.departments AS
SELECT
  department_id,
  TRIM(department) AS department
FROM instacart.departments
WHERE department_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY department) = 1;

-- Silver: products (dimension) - trimmed, deduplicated
-- Note: 1 product (id=6816) has NULL aisle_id/department_id; kept as-is for referential transparency
CREATE OR REPLACE TABLE instacart_silver.products AS
SELECT
  product_id,
  TRIM(product_name) AS product_name,
  aisle_id,
  department_id
FROM instacart.products
WHERE product_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY product_name) = 1;

-- Silver: orders (fact) - validated, type-refined
CREATE OR REPLACE TABLE instacart_silver.orders AS
SELECT
  order_id,
  user_id,
  eval_set,
  order_number,
  order_dow,
  order_hour_of_day,
  CAST(days_since_prior_order AS INT) AS days_since_prior_order
FROM instacart.orders
WHERE order_id IS NOT NULL
  AND user_id IS NOT NULL
  AND eval_set IN ('prior', 'train', 'test')
  AND order_dow BETWEEN 0 AND 6
  AND order_hour_of_day BETWEEN 0 AND 23;

-- Silver: order_products (fact) - FK-validated
CREATE OR REPLACE TABLE instacart_silver.order_products AS
SELECT
  op.order_id,
  op.product_id,
  op.add_to_cart_order,
  op.reordered
FROM instacart.order_products op
LEFT JOIN instacart_silver.orders o ON op.order_id = o.order_id
LEFT JOIN instacart_silver.products p ON op.product_id = p.product_id
WHERE op.order_id IS NOT NULL
  AND op.product_id IS NOT NULL
  AND op.reordered IN (0, 1)
  AND op.add_to_cart_order > 0;